# TFG · Lección 1 (sistemas individuales): comprobaciones en Qiskit

Cada bloque comprueba numéricamente un resultado del Capítulo 1 del cuaderno de computación cuántica.
Ejecuta las celdas **en orden**. Las salidas y figuras que obtengas son las que van al documento.

In [1]:
# Versión de Qiskit (apúntala en el cuaderno)
from qiskit import __version__
print(__version__)

2.2.3


In [2]:
import numpy as np
from numpy import sqrt
from qiskit import QuantumCircuit
from qiskit.quantum_info import Statevector, Operator
from qiskit.visualization import plot_histogram

## Experimento 1 · Estados cuánticos válidos y no válidos

Un vector es un estado cuántico si y solo si tiene norma euclídea 1.
`w = (1/3, 2/3)` tiene entradas no negativas pero no es un estado: su norma es $\sqrt{5}/3 \approx 0.745$.

In [3]:
mas = Statevector([1 / sqrt(2), 1 / sqrt(2)])      # |+>
psi = Statevector([(1 + 2j) / 3, -2 / 3])           # ejemplo del cuaderno
w   = Statevector([1 / 3, 2 / 3])                   # NO es un estado

for nombre, v in [("|+>", mas), ("psi", psi), ("w", w)]:
    print(f"{nombre:4s}  norma = {np.linalg.norm(v.data):.6f}   valido: {v.is_valid()}")

print(psi.draw("latex_source"))                      # codigo LaTeX para el cuaderno

|+>   norma = 1.000000   valido: True
psi   norma = 1.000000   valido: True
w     norma = 0.745356   valido: False


(\frac{1}{3} + \frac{2 i}{3}) |0\rangle- \frac{2}{3} |1\rangle


## Experimento 2 · Regla de Born y número de disparos

Para `psi`, la regla de Born da $\Pr(0) = 5/9 \approx 0.556$ y $\Pr(1) = 4/9 \approx 0.444$.
Simulamos la medición con distinto número de disparos (*shots*) y vemos cómo la frecuencia se acerca a la probabilidad (ley de los grandes números).
Esto anticipa el error estadístico al estimar el kernel cuántico.

In [4]:
# Probabilidades exactas (regla de Born)
print("Probabilidades exactas:", psi.probabilities_dict())

# Frecuencias con distinto numero de disparos
for shots in [100, 1000, 10000, 100000]:
    cuentas = psi.sample_counts(shots)
    f0 = cuentas.get("0", 0) / shots
    print(f"shots = {shots:6d}   frecuencia(0) = {f0:.4f}   error = {abs(f0 - 5/9):.4f}")

Probabilidades exactas: {np.str_('0'): np.float64(0.5555555555555556), np.str_('1'): np.float64(0.4444444444444444)}
shots =    100   frecuencia(0) = 0.5100   error = 0.0456
shots =   1000   frecuencia(0) = 0.5210   error = 0.0346
shots =  10000   frecuencia(0) = 0.5506   error = 0.0050
shots = 100000   frecuencia(0) = 0.5556   error = 0.0000


In [5]:
# Histograma para el cuaderno (guardalo como figura)
fig = plot_histogram(psi.sample_counts(1000))
fig.savefig("../apuntes/cuaderno_cuantica/histograma_psi.png", dpi=200, bbox_inches="tight")
fig

<Figure size 640x480 with 1 Axes>

## Experimento 3 · Fase global

Tras medir, Qiskit puede devolver $\frac{1+2i}{\sqrt 5}|0\rangle$ o $-|1\rangle$.
Comprobamos que son iguales a $|0\rangle$ y $|1\rangle$ **salvo fase global** con el método `equiv`.

In [6]:
resultado, estado = psi.measure()
print("Resultado:", resultado)
print("Estado tras medir:", estado.draw("latex_source"))

cero, uno = Statevector.from_label("0"), Statevector.from_label("1")
esperado = cero if resultado == "0" else uno
print("Igual salvo fase global:", estado.equiv(esperado))
print("Igual como vector:      ", np.allclose(estado.data, esperado.data))

Resultado: 0
Estado tras medir: (\frac{\sqrt{5}}{5} + \frac{2 \sqrt{5} i}{5}) |0\rangle
Igual salvo fase global: True
Igual como vector:       False


## Experimento 4 · Interferencia: distinguir $|+\rangle$ y $|-\rangle$

Medidos directamente, $|+\rangle$ y $|-\rangle$ dan las mismas probabilidades.
Aplicando antes $H$, se distinguen con certeza: $H|+\rangle = |0\rangle$ y $H|-\rangle = |1\rangle$.

In [7]:
H = Operator.from_label("H")
mas   = Statevector.from_label("+")
menos = Statevector.from_label("-")

print("Medida directa   |+>:", mas.probabilities_dict())
print("Medida directa   |->:", menos.probabilities_dict())
print("Tras H           |+>:", mas.evolve(H).probabilities_dict())
print("Tras H           |->:", menos.evolve(H).probabilities_dict())

Medida directa   |+>: {np.str_('0'): np.float64(0.4999999999999999), np.str_('1'): np.float64(0.4999999999999999)}
Medida directa   |->: {np.str_('0'): np.float64(0.4999999999999999), np.str_('1'): np.float64(0.4999999999999999)}
Tras H           |+>: {np.str_('0'): np.float64(0.9999999999999996)}
Tras H           |->: {np.str_('1'): np.float64(0.9999999999999996)}


## Experimento 5 · Raíz cuadrada de NOT: $(HSH)^2 = X$

Construimos el circuito $H, S, H, H, S, H$ y comparamos su matriz con la de $X$.

In [8]:
R = QuantumCircuit(1)
R.h(0); R.s(0); R.h(0)
print("Matriz de R = HSH:")
print(np.round(Operator.from_circuit(R).data, 4))

R2 = R.compose(R)                       # aplicar R dos veces
display(R2.draw(output="mpl"))
R2.draw(output="mpl").savefig("../apuntes/cuaderno_cuantica/circuito_R2.png", dpi=200, bbox_inches="tight")

X = Operator.from_label("X")
print("R^2 == X:", Operator.from_circuit(R2).equiv(X))
print(np.round(Operator.from_circuit(R2).data, 10))

Matriz de R = HSH:
[[0.5+0.5j 0.5-0.5j]
 [0.5-0.5j 0.5+0.5j]]


<Figure size 603.508x117.056 with 1 Axes>

R^2 == X: True
[[0.+0.j 1.+0.j]
 [1.+0.j 0.+0.j]]


## Experimento 6 · Las matrices son unitarias

Comprobamos $U^\dagger U = I$ para las puertas del cuaderno.

In [9]:
for nombre in ["X", "Y", "Z", "H", "S", "T"]:
    U = Operator.from_label(nombre).data
    print(f"{nombre}:  U^dagger U = I ?  {np.allclose(U.conj().T @ U, np.eye(2))}")

X:  U^dagger U = I ?  True
Y:  U^dagger U = I ?  True
Z:  U^dagger U = I ?  True
H:  U^dagger U = I ?  True
S:  U^dagger U = I ?  True
T:  U^dagger U = I ?  True
